# CARA-T-inspired retraining experiment ? Electricity

This notebook presents the Electricity experiment in the research/evaluation flow used by the Airline reference notebook: dataset and split validation, baseline description, paper-to-project mapping, validation calibration, sequential holdout policy comparison, audit, and interpretation.

**This notebook reads the saved project artifacts by default.** The final cell has an explicit opt-in to rerun the canonical evaluation script. Earlier exploratory work inspected holdout data, so do not call it a pristine untouched test set.

## 1. Scope and research question

The experiment asks when a binary electricity price-movement classifier should be updated as ordered batches arrive. The baseline is a Keras neural network over six standardized numeric features. The retraining methods are research simulations; no live service or automated deployment is implemented.

This notebook follows the supplied paper's cost-aware framing, with the Airline notebook as its organizational reference. The project-specific algorithms remain in `evaluate_test_holdout.py` and the other canonical scripts.

## 2. Imports, paths, and execution mode

Run from the repository root or this project directory. Artifact inspection is the default; setting `RUN_FULL_EVALUATION = True` in the final cell reruns the canonical evaluation and overwrites its generated result files.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "evaluate_test_holdout.py").is_file() and (base / "data" / "elec" / "processed" / "electricity_batched.csv").is_file():
            return base.resolve()
        candidate = base / "Electricity" / "electricity-cara"
        if (candidate / "evaluate_test_holdout.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate Electricity/electricity-cara from the notebook working directory.")


PROJECT_DIR = find_project_root()
DATA_PATH = PROJECT_DIR / "data" / "elec" / "processed" / "electricity_batched.csv"
MANIFEST_PATH = DATA_PATH.parent / "batch_manifest.csv"
MODEL_METADATA_PATH = PROJECT_DIR / "artifacts" / "model_v1" / "metadata.json"
HOLDOUT_DIR = PROJECT_DIR / "artifacts" / "cara_t" / "test_holdout"
SENSITIVITY_DIR = PROJECT_DIR / "artifacts" / "cara_t" / "cost_sensitivity"
print("Project:", PROJECT_DIR)
print("Saved holdout artifacts:", HOLDOUT_DIR)

## 3. Validate the processed stream and split

The project uses 100 ordered batches: 0?69 for training, 70?84 for validation, and 85?99 for sequential holdout replay. This checks the actual CSV and manifest before interpreting recorded results.

In [ ]:
for required in (DATA_PATH, MANIFEST_PATH, MODEL_METADATA_PATH):
    if not required.is_file():
        raise FileNotFoundError(f"Required Electricity input is missing: {required}")

df = pd.read_csv(DATA_PATH)
manifest = pd.read_csv(MANIFEST_PATH)
with MODEL_METADATA_PATH.open(encoding="utf-8") as file:
    model_metadata = json.load(file)

features = model_metadata["features"]
target = model_metadata["target"]
required_columns = set(features + [target, "batch_id"])
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Processed stream is missing columns: {sorted(missing)}")
if manifest["batch_id"].astype(int).tolist() != list(range(100)):
    raise ValueError("Expected manifest batch IDs 0 through 99 exactly once and in order.")
if sorted(df["batch_id"].astype(int).unique().tolist()) != list(range(100)):
    raise ValueError("Processed stream does not contain all 100 batches.")
if df[features + [target, "batch_id"]].isna().any().any():
    raise ValueError("Required stream columns contain missing values.")
if manifest["n_records"].sum() != len(df):
    raise ValueError("Manifest row counts do not reconcile with the processed stream.")

print(f"Records: {len(df):,}; features: {len(features)}; batches: {manifest.batch_id.nunique()}")
print("Model V1:", model_metadata["algorithm"])
display(manifest.groupby("split", sort=False).agg(batches=("batch_id", "count"), records=("n_records", "sum")))

## 4. Descriptive stream inspection

These summaries describe the supplied batch order and target frequencies; they are not, on their own, evidence that drift caused a change in performance.

In [ ]:
batch_summary = manifest.copy()
batch_summary["positive_rate"] = batch_summary["class_1"] / batch_summary["n_records"]
display(batch_summary.groupby("split", sort=False).agg(
    batches=("batch_id", "count"),
    records=("n_records", "sum"),
    mean_positive_rate=("positive_rate", "mean"),
    min_batch_size=("n_records", "min"),
    max_batch_size=("n_records", "max"),
).round(4))

ax = batch_summary.plot(x="batch_id", y="positive_rate", figsize=(11, 3.5), legend=False, title="Electricity positive-class rate by ordered batch")
ax.set_ylabel("Positive-class rate")
ax.set_xlabel("Batch index (ordered replay)")
plt.tight_layout()
plt.show()

## 5. Model V1 and baseline metrics

Model V1 was trained on batches 0?69. The metrics in its metadata are validation metrics from batches 70?84; the holdout Never-retrain row later in this notebook is a separate sequential holdout result. Keep those evaluations distinct.

In [ ]:
print("Training batches:", model_metadata["training_batches"])
print("Validation batches:", model_metadata["validation_batches"])
print("Training records:", f"{model_metadata['training_records']:,}")
print("Validation records:", f"{model_metadata['validation_records']:,}")
display(pd.Series(model_metadata["metrics"], name="Model V1 validation").to_frame().round(4))

## Paper method and this project's adaptation

The supplied paper formalizes the trade-off between model staleness and retraining cost (Mahadevan & Mathioudakis, 2024, Sections 4?5; the threshold variant is CARA-T, Section 5.1). Its experimental protocol uses an offline phase to select a threshold and then an online retraining strategy. This project borrows that structure, but its objective and implementation are project-specific, not a complete reproduction of the paper.

For intuition, an RBF similarity is

\[
K(q,x)=\exp(-\gamma\lVert q-x\rVert^2),
\]

and a similarity-weighted 0?1 loss proxy is

\[
C_t(M)=\operatorname{mean}_{q\in Q_t,\,(x,y)\in D_t}
       K(q,x)\,\mathbf{1}[M(x)\ne y].
\]

The implementation compares a current-batch proxy with the active model's reference proxy:

\[
\Delta_t=C_t(M_t)-C_{\mathrm{ref}}(M_t),\qquad
\Delta_t\ge\tau\;\Longrightarrow\;\text{RETRAIN},
\quad\Delta_t<\tau\;\Longrightarrow\;\text{KEEP}.
\]

Its accounting assigns the assumed \(\kappa\) to RETRAIN and the batch proxy cost to KEEP. Neither quantity is measured business loss, money, or compute. Because the loss uses true labels, decisions in a real system would wait for labels.

**Paper source:** [cost1-aware retrining algo.pdf](../../../cost1-aware%20retrining%20algo.pdf), Mahadevan & Mathioudakis, "Cost-aware retraining for machine learning," *Knowledge-Based Systems* 293 (2024), 111610. Consult Sections 4-5 for cost definitions/CARA-T and Section 7 for experimental setup. The Airline notebook is used as the project-format reference; this notebook reuses the Electricity implementation rather than copying the Airline model/data pipeline.

## 6. Validation calibration and retraining-cost sensitivity

The final holdout evaluation records the threshold selected by sequential validation replay and the `kappa` used for that run. A separate saved sensitivity experiment varies `kappa`; it is shown as a distinct analysis and should not be confused with the calibration metadata for the final holdout table.

In [ ]:
metadata_path = HOLDOUT_DIR / "experiment_metadata.json"
threshold_table_path = HOLDOUT_DIR / "validation_threshold_sensitivity.csv"
kappa_sweep_path = SENSITIVITY_DIR / "best_policy_per_kappa.csv"
for path in (metadata_path, threshold_table_path, kappa_sweep_path):
    if not path.is_file():
        raise FileNotFoundError(f"Calibration artifact is missing: {path}")

with metadata_path.open(encoding="utf-8") as file:
    experiment_metadata = json.load(file)
threshold_table = pd.read_csv(threshold_table_path)
kappa_sweep = pd.read_csv(kappa_sweep_path)
print("Final holdout calibration:", experiment_metadata["threshold_source"])
print("Selected validation threshold:", experiment_metadata["threshold"])
print("Kappa:", experiment_metadata["kappa"])
display(threshold_table.loc[threshold_table["selected"]].reset_index(drop=True))
display(kappa_sweep)

ax = kappa_sweep.plot(x="kappa", y="retrain_count", marker="o", figsize=(7, 3.5), legend=False, title="Separate validation replay: selected retrains vs assumed kappa")
ax.set_ylabel("Validation retrains")
plt.tight_layout()
plt.show()

## 7. Sequential holdout protocol

Batches 85?99 are evaluated using the active model before that batch's labels can train a replacement. A triggered update is for subsequent batches. The policy comparison reports accuracy/F1 as prediction metrics and proxy cost/retraining count as decision metrics; they answer different questions.

## 8. Recorded policy comparison

The following table is loaded from the generated holdout summary rather than hard-coded. The table represents one saved run, not a guarantee of future performance.

In [ ]:
summary_path = HOLDOUT_DIR / "test_strategy_summary.csv"
batch_results_path = HOLDOUT_DIR / "test_batch_results.csv"
if not summary_path.is_file() or not batch_results_path.is_file():
    raise FileNotFoundError("Saved holdout summary or batch results are missing; run the evaluation first.")

summary = pd.read_csv(summary_path)
batch_results = pd.read_csv(batch_results_path)
expected_strategies = {"Never retrain", "Always retrain", "Periodic", "CARA-T-inspired"}
if set(summary["strategy"]) != expected_strategies:
    raise ValueError(f"Unexpected strategy set in saved summary: {summary['strategy'].tolist()}")
if not batch_results.groupby("strategy")["batch_id"].nunique().eq(15).all():
    raise ValueError("Every Electricity strategy must cover holdout batches 85?99.")

display(summary[["strategy", "retrain_count", "mean_accuracy", "mean_f1", "total_proxy_cost"]].sort_values("strategy").round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.sort_values("total_proxy_cost").plot.bar(x="strategy", y="total_proxy_cost", ax=axes[0], legend=False, title="Total proxy cost (lower is better)")
summary.sort_values("mean_accuracy", ascending=False).plot.bar(x="strategy", y="mean_accuracy", ax=axes[1], legend=False, title="Mean holdout accuracy")
for ax in axes:
    ax.tick_params(axis="x", rotation=35)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 9. Batch-level results and audit

The audit summary is the saved implementation check for row coverage and cost reconciliation. This notebook also checks that each policy has all 15 holdout batches and that the CARA-T decisions satisfy the selected threshold.

In [ ]:
audit_path = HOLDOUT_DIR / "holdout_audit_summary.csv"
if not audit_path.is_file():
    raise FileNotFoundError(f"Holdout audit summary is missing: {audit_path}")
audit = pd.read_csv(audit_path)
required_audit = {"batch_count", "retrain_cost_matches_kappa", "summary_matches_batch_results"}
if not required_audit.issubset(audit.columns):
    raise ValueError(f"Audit summary lacks expected fields: {sorted(required_audit - set(audit.columns))}")
assert audit["batch_count"].eq(15).all()
assert audit["retrain_cost_matches_kappa"].astype(bool).all()
assert audit["summary_matches_batch_results"].astype(bool).all()

cara_rows = batch_results.loc[batch_results["strategy"] == "CARA-T-inspired"].sort_values("batch_id")
expected_decisions = np.where(cara_rows["relative_staleness"] >= experiment_metadata["threshold"], "RETRAIN", "KEEP")
assert np.array_equal(cara_rows["decision"].to_numpy(), expected_decisions)
print("PASS: all policies cover 15 batches; audit cost checks pass; CARA-T threshold decisions match.")
display(audit)

fig, ax = plt.subplots(figsize=(11, 4))
for strategy, rows in batch_results.groupby("strategy", sort=False):
    ax.plot(rows["batch_id"], rows["accuracy"], marker=".", label=strategy)
ax.set(title="Holdout accuracy by batch", xlabel="Batch ID", ylabel="Accuracy")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

## 10. Findings and limitations

In the saved holdout replay, periodic retraining has the lowest recorded proxy cost, whereas CARA-T-inspired retraining has the strongest mean accuracy and F1 by a small margin. It retrains on 14 of 15 holdout batches, so this run does not show large update savings over always retraining. These conclusions apply only to this dataset, split, model, assumptions, and recorded run.

Limitations: the objective uses assumed costs; the error-based signal needs labels; retraining uses the current batch; the result is one experiment/seed; and earlier holdout exploration means the final comparison should not be described as a completely untouched evaluation. The notebook and scripts do not implement production serving or rollback.

## 11. Reproduction

From the repository root in PowerShell, install `Electricity\electricity-cara\requirements.txt`, then run the following scripts from `Electricity\electricity-cara`. The notebook itself only reads saved results unless the final opt-in is changed.

In [ ]:
# Explicit opt-in; this reruns the canonical evaluator and regenerates holdout artifacts.
RUN_FULL_EVALUATION = False

if RUN_FULL_EVALUATION:
    subprocess.run([sys.executable, "-u", "evaluate_test_holdout.py"], cwd=PROJECT_DIR, check=True)
    print("Evaluation completed. Re-run the result-loading cells above to refresh the displayed tables.")
else:
    print("No model training was started. Set RUN_FULL_EVALUATION=True to rerun the canonical evaluation.")